In [ ]:
import re
from collections import Counter

import numpy as np
import pandas as pd


def read_ds(ds_name: str):
    with open(f'data/{ds_name}.csv') as f:
        max_actions = max(( len( str(c).split(",")) for c in f.readlines() ))
        f.seek(0)
        _names = ["util", "navigateur"] if "train" in ds_name else ["navigateur"]
        _names.extend(range(max_actions - len(_names)))
        return pd.read_csv(f, names=_names, dtype= str)

def to_categories(df, col="util"):
    df[[col]] = df[[col]].apply(lambda x: x.cat.codes)

def filter_action(value: str):
    for delim in [ "(", "<", "$", "1"]:
        if delim in value and (low_ind := value.index(delim)):
            value = value[:low_ind]
    return value

def get_outliers(df):
    df_length = df.iloc[:, 2:].notna().sum(axis=1)
    log_length = np.log10(df_length)
    q1, q3 = log_length.quantile([0.25, 0.75])
    iqr = q3 - q1
    borne_basse, borne_haute = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = (log_length < borne_basse) | (log_length > borne_haute)

    # TODO : add removing duplicate times
    return outliers

def get_sessions(df):
    debut = 2 if "util" in df.columns else 1      # le test n'a pas de colonne util
    return [[v for v in ligne if isinstance(v, str)] for ligne in df.iloc[:, debut:].to_numpy()]

def est_temps(a):
    return re.fullmatch(r"t\d+", a) is not None

def build_features(sessions, filtered_uniques):
    lignes = []
    for s in sessions:
        actions = [a for a in s if not est_temps(a)]
        temps = [int(a[1:]) for a in s if est_temps(a)]
        n = max(len(actions), 1)                   # évite une division par 0
        duree = max(temps, default=0)
        f = {"n_actions": len(actions),
             "duree": duree}
        compte = Counter(filter_action(a) for a in actions)
        for type_action in filtered_uniques:
            f["freq_" + type_action] = compte.get(type_action, 0) / n
        lignes.append(f)

    return pd.DataFrame(lignes)

pattern_ecran = re.compile(r"\((.*?)\)")
pattern_conf_ecran = re.compile(r"<(.*?)>")
pattern_chaine = re.compile(r"\$(.*?)\$")

def most_frequent(valeurs):
    # renvoie la valeur la plus fréquente de la liste, ou "aucun" si la liste est vide
    return Counter(valeurs).most_common(1)[0][0] if valeurs else "aucun"

def regex_features(sessions):
    lignes = []
    for s in sessions:
        # on applique les regex action par action
        ecrans = [m for a in s for m in pattern_ecran.findall(a)]
        confs = [m for a in s for m in pattern_conf_ecran.findall(a)]
        chaines = [m for a in s for m in pattern_chaine.findall(a)]
        lignes.append({
            "ecran_top": most_frequent(ecrans),
            "conf_top": most_frequent(confs),
            "chaine_top": most_frequent(chaines),
            "n_ecrans_distincts": len(set(ecrans)),
            "n_confs_distinctes": len(set(confs)),
            "n_chaines_distinctes": len(set(chaines)),
        })
    return pd.DataFrame(lignes)


In [25]:
features_train = read_ds("train")
features_test = read_ds("test")

features_train["util"] = pd.Categorical(features_train["util"])

uniques = features_train.iloc[:,2:].stack().dropna().unique()
filtered_uniques = list(set([filter_action(un) for un in uniques if not un.startswith("t")]))

train_outliers = get_outliers(features_train)
cleaned_features_train = features_train[~train_outliers].reset_index(drop=True)
test_outliers = get_outliers(features_test)
cleaned_features_test = features_test[~test_outliers].reset_index(drop=True)

sessions_train = get_sessions(cleaned_features_train)
sessions_test = get_sessions(cleaned_features_test)

X_train = build_features(sessions_train, filtered_uniques).reset_index(drop=True)
X_test = build_features(sessions_test, filtered_uniques).reset_index(drop=True)

X_regex_train = regex_features(sessions_train).reset_index(drop=True)
X_regex_test = regex_features(sessions_test).reset_index(drop=True)

In [29]:
nav_train = pd.get_dummies(cleaned_features_train["navigateur"], prefix="nav", dtype=int)
nav_test = pd.get_dummies(cleaned_features_test["navigateur"], prefix="nav", dtype=int)
nav_test = nav_test.reindex(columns=nav_train.columns, fill_value=0)

cat_train, cat_test = pd.DataFrame(), pd.DataFrame()
for col in ["ecran_top", "conf_top", "chaine_top"]:
    categories = pd.Categorical(X_regex_train[col]).categories
    cat_train[col] = pd.Categorical(X_regex_train[col], categories=categories).codes
    cat_test[col] = pd.Categorical(X_regex_test[col], categories=categories).codes

colonnes_num = ["n_ecrans_distincts", "n_confs_distinctes", "n_chaines_distinctes"]
X_hand_train = pd.concat([nav_train, X_train, X_regex_train[colonnes_num], cat_train], axis=1)
X_hand_test = pd.concat([nav_test, X_test, X_regex_test[colonnes_num], cat_test], axis=1)

y = cleaned_features_train["util"].cat.codes.to_numpy()

print(X_hand_train.shape, X_hand_test.shape)
print("Valeurs du test inconnues du train :", (cat_test == -1).sum().to_dict())
X_hand_train.head()


(3274, 47) (323, 47)
Valeurs du test inconnues du train : {'ecran_top': 3, 'conf_top': 2, 'chaine_top': 2}


/tmp/ipykernel_11810/2572436261.py:9: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  cat_test[col] = pd.Categorical(X_regex_test[col], categories=categories).codes


,nav_Firefox,nav_Google Chrome,nav_Microsoft Edge,nav_Opera,n_actions,duree,freq_Affichage d'une erreur,freq_Action de table,freq_Clic sur une grille d'historique de recherche,freq_Sélection d'un flag,...,freq_Exécution d'un bouton,freq_Raccourci,freq_Retour sur un écran,freq_Lancement d'une stat,n_ecrans_distincts,n_confs_distinctes,n_chaines_distinctes,ecran_top,conf_top,chaine_top
0,1,0,0,0,2514,2905,0.002387,0.004375,0.000398,0.00000,...,0.277645,0.000000,0.000796,0.033811,7,3,2,13,48,23
1,0,1,0,0,90,230,0.000000,0.000000,0.000000,0.00000,...,0.366667,0.011111,0.000000,0.000000,5,2,2,103,96,5
2,0,0,1,0,608,750,0.000000,0.009868,0.118421,0.00000,...,0.200658,0.064145,0.006579,0.000000,17,7,2,3,49,28
3,1,0,0,0,886,1445,0.001129,0.007901,0.136569,0.06772,...,0.314898,0.000000,0.019187,0.113995,10,4,1,13,48,23
4,0,1,0,0,173,275,0.000000,0.017341,0.057803,0.00000,...,0.346821,0.000000,0.000000,0.000000,8,4,2,13,67,20


In [27]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(X_hand_train, y, test_size=0.2, stratify=y, random_state=42)
print(X_tr.shape, X_val.shape)
print("Utilisateurs présents dans la validation :", len(np.unique(y_val)))

(2619, 47) (655, 47)
Utilisateurs présents dans la validation : 247


In [ ]:
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

modeles = {
    "Régression logistique": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=3000)
    ),
    "Arbre de décision": DecisionTreeClassifier(random_state=0),
    "Random Forest (100 arbres)": RandomForestClassifier(
        n_estimators=100, random_state=0, n_jobs=-1
    ),
    "Random Forest (500 arbres)": RandomForestClassifier(
        n_estimators=500, random_state=0, n_jobs=-1
    ),
}

resultats = []
for nom, modele in modeles.items():
    t0 = time.time()
    modele.fit(X_tr, y_tr)  # apprentissage sur 80 % des sessions
    pred = modele.predict(X_val)  # prédiction sur les 20 % jamais vues
    resultats.append(
        {
            "modèle": nom,
            "accuracy": accuracy_score(y_val, pred),
            "f1_macro": f1_score(y_val, pred, average="macro"),
            "temps (s)": time.time() - t0,
        }
    )

pd.DataFrame(resultats).set_index("modèle").round(3)


,accuracy,f1_macro,temps (s)
modèle,,,
Régression logistique,0.818,0.798,7.910
Arbre de décision,0.669,0.604,0.318
Random Forest (100 arbres),0.837,0.789,0.411
Random Forest (500 arbres),0.855,0.820,1.866
